# Cadence Extraction from Roll Data

This notebook analyzes IMU roll data to extract cycling cadence (pedaling rate).

## Theory

When cycling, the rider's body rocks side-to-side with each pedal stroke, creating a periodic oscillation in the roll axis. By analyzing the frequency content of this oscillation, we can estimate cadence.

**Typical cycling cadence:** 60-100 RPM (1-1.67 Hz)

## Approach

1. Load VTX file and extract roll data
2. Apply bandpass filter to isolate cadence frequencies (0.5-3 Hz)
3. Use sliding window FFT to compute instantaneous cadence
4. Smooth and validate results
5. Compare with known ground truth (if available)

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.fft import rfft, rfftfreq
import pandas as pd
sys.path.append('../packages/vtx-parser/python')
from vtx_parser import decode_vtx

# Matplotlib settings for better plots
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10
%matplotlib inline

## 1. Load VTX Data

In [ ]:
# Load the VTX file
vtx_file = 'data/Bridge and hawk.vtx'

with open(vtx_file, 'rb') as f:
    data = decode_vtx(f.read())

samples = data.records
print(f"Loaded {len(samples)} samples")
print(f"Duration: {(samples[-1].timestamp - samples[0].timestamp) / 1000:.1f} seconds")

# Extract timestamps and roll data
timestamps_ms = np.array([s.timestamp for s in samples])
timestamps_s = (timestamps_ms - timestamps_ms[0]) / 1000.0  # Relative time in seconds

roll = np.array([s.roll if hasattr(s, 'roll') and s.roll is not None else 0 for s in samples])
pitch = np.array([s.pitch if hasattr(s, 'pitch') and s.pitch is not None else 0 for s in samples])
yaw = np.array([s.yaw if hasattr(s, 'yaw') and s.yaw is not None else 0 for s in samples])

# Calculate sample rate
dt = np.mean(np.diff(timestamps_s))
sample_rate = 1.0 / dt
print(f"Sample rate: {sample_rate:.2f} Hz")
print(f"Roll data range: [{roll.min():.1f}, {roll.max():.1f}] degrees")
print(f"Roll std dev: {roll.std():.2f} degrees")

## 2. Visualize Raw Roll Data

In [ ]:
# Plot first 60 seconds to see oscillations
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Full recording
axes[0].plot(timestamps_s / 60, roll, linewidth=0.5, alpha=0.7)
axes[0].set_xlabel('Time (minutes)')
axes[0].set_ylabel('Roll (degrees)')
axes[0].set_title('Full Roll Data')
axes[0].grid(True, alpha=0.3)

# Zoomed view - first 60 seconds
mask_60s = timestamps_s <= 60
axes[1].plot(timestamps_s[mask_60s], roll[mask_60s], linewidth=1)
axes[1].set_xlabel('Time (seconds)')
axes[1].set_ylabel('Roll (degrees)')
axes[1].set_title('Roll Data - First 60 Seconds (Look for periodic oscillations)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Frequency Analysis - What frequencies are present?

Let's look at the frequency spectrum to see if cadence frequencies (1-1.67 Hz for 60-100 RPM) are visible.

In [ ]:
# Compute power spectral density
freqs, psd = signal.welch(roll, fs=sample_rate, nperseg=min(1024, len(roll)//4))

fig, ax = plt.subplots(figsize=(14, 6))
ax.semilogy(freqs, psd, linewidth=1)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Power Spectral Density')
ax.set_title('Roll Data - Frequency Spectrum')
ax.set_xlim(0, 5)  # Focus on 0-5 Hz (cadence range)
ax.axvspan(0.5, 2.0, alpha=0.2, color='green', label='Typical cadence range (30-120 RPM)')
ax.grid(True, alpha=0.3)
ax.legend()

# Find dominant frequency in cadence range
cadence_mask = (freqs >= 0.5) & (freqs <= 2.0)
if np.any(cadence_mask):
    peak_freq = freqs[cadence_mask][np.argmax(psd[cadence_mask])]
    peak_cadence = peak_freq * 60  # Convert Hz to RPM
    ax.axvline(peak_freq, color='red', linestyle='--', linewidth=2, label=f'Peak: {peak_freq:.2f} Hz ({peak_cadence:.0f} RPM)')
    ax.legend()
    print(f"Dominant frequency in cadence range: {peak_freq:.2f} Hz ({peak_cadence:.0f} RPM)")

plt.tight_layout()
plt.show()

## 4. Bandpass Filter - Isolate Cadence Frequencies

Apply a bandpass filter to remove drift (low frequencies) and noise (high frequencies).

In [ ]:
# Design bandpass filter for cadence range
# Cadence: 30-120 RPM = 0.5-2.0 Hz
lowcut = 0.5   # 30 RPM
highcut = 2.5  # 150 RPM (wider range to catch everything)
order = 4

# Butterworth bandpass filter
sos = signal.butter(order, [lowcut, highcut], btype='band', fs=sample_rate, output='sos')
roll_filtered = signal.sosfiltfilt(sos, roll)

# Plot comparison
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Show 30 second window for detail
mask_30s = (timestamps_s >= 60) & (timestamps_s <= 90)

axes[0].plot(timestamps_s[mask_30s], roll[mask_30s], linewidth=1, alpha=0.7, label='Raw roll')
axes[0].set_ylabel('Roll (degrees)')
axes[0].set_title('Raw Roll Data (60-90s window)')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(timestamps_s[mask_30s], roll_filtered[mask_30s], linewidth=1.5, color='green', label='Filtered (0.5-2.5 Hz)')
axes[1].set_xlabel('Time (seconds)')
axes[1].set_ylabel('Roll (degrees)')
axes[1].set_title('Bandpass Filtered Roll (Cadence Oscillations)')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Filtered roll std dev: {roll_filtered.std():.3f} degrees")

## 5. Instantaneous Cadence - Sliding Window Analysis

Use a sliding window to compute cadence over time. This captures changes in pedaling rate.

In [ ]:
def compute_instantaneous_cadence(signal_data, sample_rate, window_seconds=10, step_seconds=1):
    """
    Compute instantaneous cadence using sliding window FFT.
    
    Args:
        signal_data: Filtered roll data
        sample_rate: Sampling rate (Hz)
        window_seconds: Window length for FFT (default 10s)
        step_seconds: Step size between windows (default 1s)
    
    Returns:
        times: Array of center times for each window
        cadences: Array of estimated cadence (RPM) for each window
        confidences: Array of confidence scores (peak power / total power)
    """
    window_samples = int(window_seconds * sample_rate)
    step_samples = int(step_seconds * sample_rate)
    
    times = []
    cadences = []
    confidences = []
    
    for start_idx in range(0, len(signal_data) - window_samples, step_samples):
        end_idx = start_idx + window_samples
        window_data = signal_data[start_idx:end_idx]
        
        # Compute FFT
        freqs = rfftfreq(len(window_data), 1.0/sample_rate)
        fft_values = np.abs(rfft(window_data))
        
        # Focus on cadence range
        cadence_mask = (freqs >= 0.5) & (freqs <= 2.5)
        
        if np.any(cadence_mask):
            cadence_freqs = freqs[cadence_mask]
            cadence_power = fft_values[cadence_mask]
            
            # Find peak frequency
            peak_idx = np.argmax(cadence_power)
            peak_freq = cadence_freqs[peak_idx]
            peak_power = cadence_power[peak_idx]
            
            # Confidence: ratio of peak to total power in cadence range
            total_power = np.sum(cadence_power)
            confidence = peak_power / total_power if total_power > 0 else 0
            
            # Convert to RPM
            cadence_rpm = peak_freq * 60
            
            # Store results (use center time of window)
            center_time = (start_idx + window_samples // 2) / sample_rate
            times.append(center_time)
            cadences.append(cadence_rpm)
            confidences.append(confidence)
    
    return np.array(times), np.array(cadences), np.array(confidences)

# Compute cadence
print("Computing instantaneous cadence...")
cadence_times, cadence_rpm, cadence_confidence = compute_instantaneous_cadence(
    roll_filtered, 
    sample_rate, 
    window_seconds=10,  # 10 second window for stable frequency estimate
    step_seconds=2      # Update every 2 seconds
)

print(f"Computed {len(cadence_rpm)} cadence estimates")
print(f"Mean cadence: {cadence_rpm.mean():.1f} RPM")
print(f"Cadence range: [{cadence_rpm.min():.1f}, {cadence_rpm.max():.1f}] RPM")
print(f"Mean confidence: {cadence_confidence.mean():.3f}")

## 6. Visualize Cadence Over Time

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Plot 1: Raw roll data
axes[0].plot(timestamps_s / 60, roll, linewidth=0.5, alpha=0.5, color='gray', label='Raw roll')
axes[0].plot(timestamps_s / 60, roll_filtered, linewidth=0.8, color='blue', label='Filtered roll')
axes[0].set_ylabel('Roll (degrees)')
axes[0].set_title('Roll Data')
axes[0].grid(True, alpha=0.3)
axes[0].legend(loc='upper right')

# Plot 2: Estimated cadence
axes[1].plot(cadence_times / 60, cadence_rpm, linewidth=2, color='green', marker='o', markersize=3, label='Estimated cadence')
axes[1].axhspan(60, 100, alpha=0.1, color='green', label='Typical cadence range')
axes[1].set_ylabel('Cadence (RPM)')
axes[1].set_title('Estimated Cadence from Roll Oscillations')
axes[1].set_ylim(0, 150)
axes[1].grid(True, alpha=0.3)
axes[1].legend(loc='upper right')

# Plot 3: Confidence score
axes[2].plot(cadence_times / 60, cadence_confidence, linewidth=2, color='orange', marker='o', markersize=3)
axes[2].axhline(0.3, color='red', linestyle='--', linewidth=1, alpha=0.5, label='Low confidence threshold')
axes[2].set_xlabel('Time (minutes)')
axes[2].set_ylabel('Confidence')
axes[2].set_title('Cadence Estimate Confidence (higher = more periodic signal)')
axes[2].set_ylim(0, 1)
axes[2].grid(True, alpha=0.3)
axes[2].legend(loc='upper right')

plt.tight_layout()
plt.savefig('cadence_analysis.png', dpi=150, bbox_inches='tight')
print("✓ Saved plot to: cadence_analysis.png")
plt.show()

## 7. Statistical Summary

In [ ]:
# Filter by confidence threshold
confidence_threshold = 0.2
high_confidence_mask = cadence_confidence > confidence_threshold

if np.any(high_confidence_mask):
    cadence_filtered = cadence_rpm[high_confidence_mask]
    
    print("\n" + "="*70)
    print("CADENCE ANALYSIS SUMMARY")
    print("="*70)
    print(f"\nTotal recording duration: {timestamps_s[-1] / 60:.1f} minutes")
    print(f"Total cadence estimates: {len(cadence_rpm)}")
    print(f"High confidence estimates (>{confidence_threshold}): {np.sum(high_confidence_mask)} ({100*np.sum(high_confidence_mask)/len(cadence_rpm):.1f}%)")
    
    print(f"\n{'─'*70}")
    print("All Estimates:")
    print(f"  Mean cadence: {cadence_rpm.mean():.1f} RPM")
    print(f"  Median cadence: {np.median(cadence_rpm):.1f} RPM")
    print(f"  Std dev: {cadence_rpm.std():.1f} RPM")
    print(f"  Range: [{cadence_rpm.min():.1f}, {cadence_rpm.max():.1f}] RPM")
    
    print(f"\n{'─'*70}")
    print(f"High Confidence Estimates (>{confidence_threshold}):")
    print(f"  Mean cadence: {cadence_filtered.mean():.1f} RPM")
    print(f"  Median cadence: {np.median(cadence_filtered):.1f} RPM")
    print(f"  Std dev: {cadence_filtered.std():.1f} RPM")
    print(f"  Range: [{cadence_filtered.min():.1f}, {cadence_filtered.max():.1f}] RPM")
    
    # Check if it's reasonable
    reasonable = (cadence_filtered.mean() >= 40) and (cadence_filtered.mean() <= 120)
    print(f"\n{'─'*70}")
    print(f"Assessment: {'✅ REASONABLE' if reasonable else '⚠️  QUESTIONABLE'}")
    
    if reasonable:
        print("\nThe estimated cadence falls within typical cycling range (40-120 RPM).")
        print("This suggests roll oscillations DO contain pedaling cadence information!")
    else:
        print("\nThe estimated cadence is outside typical cycling range.")
        print("This could mean:")
        print("  1. Roll oscillations are dominated by road vibration, not pedaling")
        print("  2. The algorithm needs tuning (different frequency range or window size)")
        print("  3. The rider was coasting or stopped frequently")
    
    print("\n" + "="*70)
else:
    print("No high-confidence cadence estimates found.")
    print("Roll data may be too noisy or lack periodic pedaling signal.")

## 8. Export Results

Save cadence estimates for further analysis or comparison with ground truth.

In [ ]:
# Create DataFrame
df = pd.DataFrame({
    'time_seconds': cadence_times,
    'time_minutes': cadence_times / 60,
    'cadence_rpm': cadence_rpm,
    'confidence': cadence_confidence
})

# Save to CSV
output_file = 'cadence_estimates.csv'
df.to_csv(output_file, index=False)
print(f"✓ Saved cadence estimates to: {output_file}")

# Show first few rows
print("\nFirst 10 estimates:")
print(df.head(10).to_string(index=False))

## 9. Future Improvements

This notebook provides a baseline for cadence extraction from roll data. Future enhancements:

1. **Multi-axis fusion**: Combine roll, pitch, and gyro data for more robust estimates
2. **Machine learning**: Train a model on labeled data (IMU + power meter cadence)
3. **Peak detection**: Use zero-crossings or peak detection instead of FFT
4. **Kalman filtering**: Apply sensor fusion to smooth estimates
5. **Stopped detection**: Identify when rider is coasting/stopped and exclude from cadence
6. **Gyro integration**: Z-axis gyro may have stronger pedaling signal

## Notes for Testing with Different Damping

When testing new damping materials, compare:
- **Confidence scores**: Higher = more periodic signal, less noise
- **Cadence stability**: Lower std dev = more consistent estimates
- **Valid estimate percentage**: More high-confidence windows = better signal quality